# Pitch arsenal optimization

Hiring walkthrough for this repo’s optimization stack. Narrative:

1. **Data** — load MLB Statcast pitches
2. **Features** — prepare context + pitch-choice columns
3. **Outcome model** — predict run value / pitch xwOBA
4. **Optimize** — constrained usage mix vs actual
5. **Recommendations** — markdown report for one pitcher

All modeling code lives in one module: `pitch_dataset.arsenal` (`src/pitch_dataset/arsenal.py`).

**Demo window:** MLB 2026-03-25 → 2026-08-13 (~491k pitches, season-to-date).

## 1. Data

Parquet is local/gitignored. Pull the same window with:

```bash
uv run pitch-dataset pull --league mlb --season 2026 --start 2026-03-20 --end 2026-08-14
```

In [ ]:
from pathlib import Path

from pitch_dataset.arsenal import (
    CONTEXT_FEATURE_COLS,
    format_recommendation_report,
    load_outcome_model,
    optimize_pitcher,
    prepare_pitches,
    train_outcome_model,
)
from pitch_dataset.storage import read_pitches

data_path = Path("../data/pitches_mlb_2026.parquet")
model_path = Path("../models/outcome_model.joblib")
pitches = read_pitches(data_path)
print(len(pitches), pitches["game_date"].min(), pitches["game_date"].max())
pitches[["player_name", "pitch_type", "balls", "strikes", "stand", "p_throws"]].head()

## 2. Features

`prepare_pitches` builds modeling columns: platoon, count buckets, zone/location,
TTO, runners, batter prior, previous-pitch flags, and arsenal pairing signals.

In [ ]:
prepared = prepare_pitches(pitches)
print("rows after filters:", len(prepared))
print("context features:", len(CONTEXT_FEATURE_COLS))
prepared[CONTEXT_FEATURE_COLS[:8] + ["pitch_type", "target_rv", "target_xwoba"]].head()

## 3. Outcome model

Dual `HistGradientBoostingRegressor` targets: `delta_run_exp` (RV) and constructed pitch xwOBA.
Reuse the committed artifact when present; otherwise train.

In [ ]:
if model_path.exists():
    model = load_outcome_model(model_path)
    print("loaded", model_path)
    print(model.meta)
else:
    model, metrics = train_outcome_model(prepared, model_path=model_path)
    print(metrics)

## 4–5. Optimize + recommendations

For each platoon (and count) segment, solve a constrained mix that minimizes expected
pitch xwOBA subject to min/max % and max shift from current usage.

In [ ]:
rec = optimize_pitcher(prepared, model, pitcher="Cease")
print(format_recommendation_report(rec))

## Method (short)

1. **Outcome model:** `HistGradientBoostingRegressor` predicts pitch-level run value (`delta_run_exp`) and an approximate pitch xwOBA from context + pitch type.
2. **Context features:** platoon, count, zone/location, TTO, baserunners, batter prior xwOBA, previous pitch, and arsenal pairing (velo/movement separation, release similarity).
3. **Counterfactuals:** for each pitcher × platoon (and count) segment, solve a constrained mix that minimizes expected xwOBA subject to min/max % and max shift from current usage.
4. **Limitations:** early-season sample; holds location fixed; ignores catcher/game-planning constraints; pitch xwOBA for non-BIP is a heuristic.

CLI equivalents: `uv run pitch-dataset train-model` and `uv run pitch-dataset optimize --pitcher Cease`.